# ARC vision-to-grid SFT

This notebook trains one Qwen3-VL 4B LoRA adapter on four L4 GPUs. Each episode presents the full ARC task as one image plus exact numeric grids, and the model learns to return only `{"output":[[...]]}`.

The 4,308 training episodes come from real ARC training tasks: leave-one-demonstration-out episodes plus official training queries. The 172 official evaluation queries are untouched by training and scored by exact grid equality. There is no rotation or color-permutation multiplication.

Run with Kaggle's **4 x L4** accelerator. Internet remains disabled.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
SMOKE_TEST = True
os.environ["ARC_VLM_SMOKE_TEST"] = "1" if SMOKE_TEST else "0"

WHEELHOUSE = Path("/kaggle/input/datasets/aishikai/offline-unsloth-trl-wheelhouse-py312-cu128")
requirements = WHEELHOUSE / "requirements.in"
assert requirements.exists(), "Attach aishikai/offline-unsloth-trl-wheelhouse-py312-cu128"
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-index",
    "--find-links", str(WHEELHOUSE), "-r", str(requirements),
], check=True)
missing = [name for name in ("unsloth", "trl", "datasets", "tensorboard") if importlib.util.find_spec(name) is None]
assert not missing, f"Offline dependency installation failed: {missing}"
print("Offline dependencies installed.")


In [ ]:
from pathlib import Path


def find_parent(filename, sibling=None, preferred=()):
    matches = []
    for path in Path("/kaggle/input").glob(f"**/{filename}"):
        if sibling and not (path.parent / sibling).exists():
            continue
        matches.append((-sum(part in str(path).lower() for part in preferred), len(path.parts), path.parent))
    return sorted(matches)[0][2] if matches else None


COMPETITION_DIR = find_parent(
    "arc-agi_training_challenges.json",
    sibling="arc-agi_evaluation_solutions.json",
    preferred=("arc-prize-2026",),
)
MODEL_PATH = find_parent(
    "config.json",
    sibling="model.safetensors.index.json",
    preferred=("qwen3-vl-4b", "bnb-4bit"),
)
if MODEL_PATH is None:
    MODEL_PATH = find_parent("config.json", preferred=("qwen3-vl-4b", "bnb-4bit"))
assert COMPETITION_DIR, "Attach the ARC Prize 2026 competition data"
assert MODEL_PATH, "Attach aishikai/qwen3-vl-4b-instruct-unsloth-bnb-4bit"
Path("data").mkdir(exist_ok=True)
Path("data/model_path.txt").write_text(str(MODEL_PATH))
print({"competition": str(COMPETITION_DIR), "model": str(MODEL_PATH)})


In [ ]:
%%writefile arc_vlm_data.py
"""Build rendered, exact-grid ARC vision SFT episodes."""

import json
from pathlib import Path

from datasets import Dataset, Image, Sequence
from PIL import Image as PILImage, ImageDraw, ImageFont


PALETTE = [
    (0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64), (255, 220, 0),
    (170, 170, 170), (240, 18, 190), (255, 133, 27), (127, 219, 255), (135, 12, 37),
]


def validate_grid(grid):
    if not isinstance(grid, list) or not grid or not isinstance(grid[0], list) or not grid[0]:
        raise ValueError("grid must be a non-empty list of rows")
    width = len(grid[0])
    if len(grid) > 30 or width > 30 or any(len(row) != width for row in grid):
        raise ValueError("grid must be rectangular and at most 30x30")
    if any(not isinstance(value, int) or not 0 <= value <= 9 for row in grid for value in row):
        raise ValueError("grid colors must be integers 0..9")
    return grid


def grid_text(grid):
    validate_grid(grid)
    return "\n".join("".join(str(value) for value in row) for row in grid)


def task_prompt(demos, query):
    parts = [
        "Infer the exact output for QUERY. Use the image for whole-scene spatial structure and the exact grids below for coordinates and colors.",
    ]
    for index, example in enumerate(demos, 1):
        inp, out = validate_grid(example["input"]), validate_grid(example["output"])
        parts.append(f"DEMO {index} INPUT ({len(inp)}x{len(inp[0])}):\n{grid_text(inp)}")
        parts.append(f"DEMO {index} OUTPUT ({len(out)}x{len(out[0])}):\n{grid_text(out)}")
    query = validate_grid(query)
    parts.append(f"QUERY INPUT ({len(query)}x{len(query[0])}):\n{grid_text(query)}")
    parts.append('Return only JSON in the form {"output":[[...]]}.')
    return "\n\n".join(parts)


def _grid_image(grid, size=210):
    grid = validate_grid(grid)
    height, width = len(grid), len(grid[0])
    cell = max(3, min(size // height, size // width))
    image = PILImage.new("RGB", (width * cell + 1, height * cell + 1), (64, 64, 64))
    draw = ImageDraw.Draw(image)
    for row, values in enumerate(grid):
        for col, value in enumerate(values):
            x, y = col * cell, row * cell
            draw.rectangle((x, y, x + cell - 1, y + cell - 1), fill=PALETTE[value])
    return image


def _pair_panel(label, inp, out=None):
    font = ImageFont.load_default()
    panel = PILImage.new("RGB", (500, 255), "white")
    draw = ImageDraw.Draw(panel)
    draw.text((8, 8), label, fill="black", font=font)
    draw.text((8, 28), "INPUT", fill="black", font=font)
    left = _grid_image(inp)
    panel.paste(left, (8, 45))
    if out is not None:
        draw.text((258, 28), "OUTPUT", fill="black", font=font)
        right = _grid_image(out)
        panel.paste(right, (258, 45))
    else:
        draw.text((258, 100), "?", fill="black", font=font)
    return panel


def render_task(demos, query):
    panels = [_pair_panel(f"DEMO {index}", row["input"], row["output"]) for index, row in enumerate(demos, 1)]
    panels.append(_pair_panel("QUERY", query))
    rows = (len(panels) + 1) // 2
    image = PILImage.new("RGB", (1010, rows * 265), (224, 224, 224))
    for index, panel in enumerate(panels):
        image.paste(panel, ((index % 2) * 510, (index // 2) * 265))
    if max(image.size) > 1280:
        scale = 1280 / max(image.size)
        image = image.resize((round(image.width * scale), round(image.height * scale)), PILImage.Resampling.NEAREST)
    return image


def episode(task_id, episode_id, demos, query, output, image_dir):
    output = validate_grid(output)
    path = image_dir / f"{task_id}_{episode_id}.png"
    render_task(demos, query).save(path, optimize=True)
    return {
        "task_id": task_id,
        "episode_id": episode_id,
        "images": [str(path)],
        "prompt": [{
            "role": "user",
            "content": [
                {"type": "image", "image": str(path)},
                {"type": "text", "text": task_prompt(demos, query)},
            ],
        }],
        "completion": [{
            "role": "assistant",
            "content": [{"type": "text", "text": json.dumps({"output": output}, separators=(",", ":"))}],
        }],
        "output": output,
    }


def build_split(challenges, solutions, image_dir, leave_one_out):
    image_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for task_id, task in challenges.items():
        train = task["train"]
        if leave_one_out:
            for index, held_out in enumerate(train):
                demos = train[:index] + train[index + 1:]
                rows.append(episode(task_id, f"demo_{index}", demos, held_out["input"], held_out["output"], image_dir))
        for index, test in enumerate(task["test"]):
            rows.append(episode(task_id, f"test_{index}", train, test["input"], solutions[task_id][index], image_dir))
    return rows


def build_datasets(competition_dir, output_dir):
    competition_dir, output_dir = Path(competition_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    load = lambda name: json.loads((competition_dir / name).read_text())
    train_rows = build_split(
        load("arc-agi_training_challenges.json"), load("arc-agi_training_solutions.json"),
        output_dir / "images/train", leave_one_out=True,
    )
    eval_rows = build_split(
        load("arc-agi_evaluation_challenges.json"), load("arc-agi_evaluation_solutions.json"),
        output_dir / "images/eval", leave_one_out=False,
    )
    for name, rows in (("train", train_rows), ("eval", eval_rows)):
        dataset = Dataset.from_list(rows).cast_column("images", Sequence(Image()))
        dataset.save_to_disk(output_dir / name)
    return len(train_rows), len(eval_rows)


In [ ]:
from arc_vlm_data import build_datasets

train_count, eval_count = build_datasets(COMPETITION_DIR, "data")
assert (train_count, eval_count) == (4308, 172), (train_count, eval_count)
print({"train_episodes": train_count, "held_out_queries": eval_count})


In [ ]:
%%writefile gaslamp_callback.py
import os
import json
import threading
import time
from http.server import HTTPServer, BaseHTTPRequestHandler
from transformers import TrainerCallback, TrainingArguments, TrainerState, TrainerControl
from typing import Dict, Any, Optional

# ─── Global State ────────────────────────────────────────────────────
# Shared in-memory state accessible by the HTTP handler thread.
_GLOBAL_PAYLOAD = {
    "meta": {"max_steps": 0, "total_epochs": 0, "task_type": "sft"},
    "hardware": {},
    "hyperparameters": {},
    "logs": [],
    "phase": "idle",           # idle | training | completed | error
    "elapsed_seconds": 0,
    "eta_seconds": None,
}

# SSE subscribers: list of threading.Event objects to notify on new data
_SSE_SUBSCRIBERS = []
_SSE_LOCK = threading.Lock()


def _notify_subscribers():
    """Wake up all SSE subscriber threads so they push the latest payload."""
    with _SSE_LOCK:
        for event in _SSE_SUBSCRIBERS:
            event.set()


# ─── HTTP Request Handler ────────────────────────────────────────────
class DashboardRequestHandler(BaseHTTPRequestHandler):
    """Serves the dashboard HTML, JSON metrics, health check, and SSE stream."""

    template_path = "templates/dashboard.html"

    def do_GET(self):
        try:
            if self.path == "/":
                self._serve_html()
            elif self.path == "/api/metrics":
                self._serve_metrics()
            elif self.path == "/api/health":
                self._serve_health()
            elif self.path == "/api/stream":
                self._serve_sse()
            else:
                self.send_response(404)
                self.end_headers()
        except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError):
            pass

    def _serve_html(self):
        self.send_response(200)
        self.send_header("Content-type", "text/html; charset=utf-8")
        self.end_headers()
        if os.path.exists(self.template_path):
            with open(self.template_path, "rb") as f:
                self.wfile.write(f.read())
        else:
            self.wfile.write(
                b"<h1>Gaslamp Dashboard</h1>"
                b"<p>Error: templates/dashboard.html not found locally.</p>"
            )

    def _serve_metrics(self):
        self.send_response(200)
        self.send_header("Content-type", "application/json")
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        try:
            self.wfile.write(json.dumps(_GLOBAL_PAYLOAD).encode("utf-8"))
        except Exception as e:
            self.wfile.write(json.dumps({"error": str(e)}).encode("utf-8"))

    def _serve_health(self):
        self.send_response(200)
        self.send_header("Content-type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps({"status": "healthy"}).encode("utf-8"))

    def _serve_sse(self):
        """Server-Sent Events endpoint. Holds the connection open and pushes updates."""
        self.send_response(200)
        self.send_header("Content-type", "text/event-stream")
        self.send_header("Cache-Control", "no-cache")
        self.send_header("Connection", "keep-alive")
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()

        # Tell the browser to retry every 3 seconds if the connection drops
        self.wfile.write(b"retry: 3000\n\n")
        self.wfile.flush()

        # Register this connection as a subscriber
        notify_event = threading.Event()
        with _SSE_LOCK:
            _SSE_SUBSCRIBERS.append(notify_event)

        try:
            # Send the current full state immediately on connect
            self._send_sse_event(notify_event)

            # Then wait for updates
            while True:
                # Block until new data arrives (or timeout for heartbeat)
                got_data = notify_event.wait(timeout=15)
                if got_data:
                    notify_event.clear()
                    self._send_sse_event(notify_event)
                else:
                    # Heartbeat to keep connection alive
                    self.wfile.write(b": heartbeat\n\n")
                    self.wfile.flush()

        except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError, OSError):
            pass
        finally:
            with _SSE_LOCK:
                if notify_event in _SSE_SUBSCRIBERS:
                    _SSE_SUBSCRIBERS.remove(notify_event)

    def _send_sse_event(self, notify_event):
        """Serialize the current payload as an SSE event."""
        payload = _GLOBAL_PAYLOAD
        step = 0
        logs = payload.get("logs", [])
        if logs:
            last = logs[-1]
            step = last.get("step", 0)

        data = json.dumps(payload)
        msg = f"id: {step}\nevent: progress\ndata: {data}\n\n"
        self.wfile.write(msg.encode("utf-8"))
        self.wfile.flush()

    def log_message(self, format, *args):
        # Suppress HTTP access logs
        pass


# ─── Trainer Callback ────────────────────────────────────────────────
class GaslampDashboardCallback(TrainerCallback):
    """
    HuggingFace TrainerCallback that spawns a local HTTP server with:
      - GET /           → Dashboard HTML
      - GET /api/metrics → Full JSON payload (for reconnection recovery)
      - GET /api/stream  → SSE live stream (instant push updates)
      - GET /api/health  → Health check

    Args:
        port      : HTTP port for the dashboard (default 8080).
        task_type : One of "sft", "dpo", "grpo", "vision" — controls which
                    panels the dashboard renders.
    """

    def __init__(self, port: int = 8080, task_type: str = "sft"):
        self.port = port
        self.task_type = task_type.lower()
        self.server_thread = None
        self.httpd = None
        self.is_running = False
        self._train_start_time = None
        # Memory baseline — captured once before training starts
        self._baseline_vram_mb = 0
        self._total_vram_mb = 0
        self._train_runtime_seconds = None

    def on_train_begin(self, args: TrainingArguments, state: TrainerState, control: TrainerControl, **kwargs):
        """Starts the background HTTP daemon server."""
        global _GLOBAL_PAYLOAD
        if state.is_world_process_zero and not self.is_running:
            self._train_start_time = time.time()
            # Snapshot GPU memory *before* training begins (model load baseline)
            try:
                import torch
                if torch.cuda.is_available():
                    self._baseline_vram_mb = int(torch.cuda.max_memory_reserved() / (1024 * 1024))
                    props = torch.cuda.get_device_properties(0)
                    self._total_vram_mb = int(props.total_memory / (1024 * 1024))
                elif torch.backends.mps.is_available():
                    self._baseline_vram_mb = int(torch.mps.driver_allocated_memory() / (1024 * 1024))
                    self._total_vram_mb = int(torch.mps.recommended_max_memory() / (1024 * 1024))
            except Exception:
                pass
            _GLOBAL_PAYLOAD["phase"] = "training"
            _GLOBAL_PAYLOAD["meta"]["task_type"] = self.task_type
            self._start_server()

    def _start_server(self):
        """Spawns the local web server on a background daemon thread."""
        try:
            self.httpd = HTTPServer(("localhost", self.port), DashboardRequestHandler)
            self.server_thread = threading.Thread(target=self.httpd.serve_forever, daemon=True)
            self.server_thread.start()
            self.is_running = True

            # Print a prominent, clickable log in the user's terminal
            time.sleep(0.5)
            print("\n" + "=" * 60)
            print(f"🚀 Gaslamp Live Training Dashboard: http://localhost:{self.port}/")
            print("=" * 60 + "\n")
        except OSError as e:
            if e.errno == 48:  # Address already in use
                print(f"⚠️ [Gaslamp] Port {self.port} is busy. Dashboard might already be running.")
            else:
                print(f"⚠️ [Gaslamp] Failed to start web server: {e}")
        except Exception as e:
            print(f"⚠️ [Gaslamp] Failed to start web server: {e}")

    def on_train_end(self, args: TrainingArguments, state: TrainerState, control: TrainerControl, **kwargs):
        """Marks training as completed."""
        global _GLOBAL_PAYLOAD
        if state.is_world_process_zero:
            elapsed = 0
            if self._train_start_time:
                elapsed = round(time.time() - self._train_start_time, 1)
            _GLOBAL_PAYLOAD["phase"] = "completed"
            _GLOBAL_PAYLOAD["elapsed_seconds"] = elapsed
            _GLOBAL_PAYLOAD["eta_seconds"] = 0
            # Final memory summary (mirrors unsloth-studio Colab cell 12)
            try:
                import torch
                peak_mb = lora_mb = total_mb = None
                if torch.cuda.is_available():
                    peak_mb  = int(torch.cuda.max_memory_reserved() / (1024 * 1024))
                    total_mb = self._total_vram_mb or 1
                elif torch.backends.mps.is_available():
                    peak_mb  = int(torch.mps.driver_allocated_memory() / (1024 * 1024))
                    total_mb = self._total_vram_mb or 1
                if peak_mb is not None:
                    lora_mb = peak_mb - self._baseline_vram_mb
                    _GLOBAL_PAYLOAD.setdefault("hardware", {})
                    _GLOBAL_PAYLOAD["hardware"].update({
                        "peak_vram_mb":      peak_mb,
                        "baseline_vram_mb":  self._baseline_vram_mb,
                        "lora_vram_mb":      max(lora_mb, 0),
                        "total_vram_mb":     self._total_vram_mb,
                        "vram_pct":          round(peak_mb / total_mb * 100, 1),
                        "lora_vram_pct":     round(max(lora_mb, 0) / total_mb * 100, 1),
                    })
            except Exception:
                pass
            _GLOBAL_PAYLOAD["train_runtime_seconds"] = elapsed
            _notify_subscribers()

    def on_log(self, args: TrainingArguments, state: TrainerState, control: TrainerControl, logs: Dict[str, float] = None, **kwargs):
        """Called whenever `trainer.log()` is invoked. Updates payload + notifies SSE subscribers."""
        global _GLOBAL_PAYLOAD

        if state.is_world_process_zero:
            # 1. Hardware stats
            hw_info = {"device": "Unknown", "peak_vram_mb": 0}
            try:
                import torch
                if torch.cuda.is_available():
                    hw_info["device"] = torch.cuda.get_device_name(0)
                    hw_info["peak_vram_mb"] = int(torch.cuda.max_memory_allocated() / (1024 * 1024))
                elif torch.backends.mps.is_available():
                    hw_info["device"] = "Apple MPS"
                    hw_info["peak_vram_mb"] = int(torch.mps.driver_allocated_memory() / (1024 * 1024))
            except Exception:
                pass

            # CPU RAM (optional, guarded)
            try:
                import psutil
                hw_info["cpu_ram_mb"] = int(psutil.Process().memory_info().rss / (1024 * 1024))
            except Exception:
                pass

            # 2. Hyperparameters
            hp_info = {}
            try:
                batch = getattr(args, "per_device_train_batch_size", None) \
                     or getattr(args, "train_batch_size", 1)
                hp_info = {
                    "learning_rate":               args.learning_rate,
                    "per_device_train_batch_size": batch,
                    "gradient_accumulation":       args.gradient_accumulation_steps,
                    "optimizer":                   getattr(args, "optim", "Unknown"),
                    "seed":                        getattr(args, "seed", None),
                }
                # GRPO: also surface generation_batch_size if present
                gen_bs = getattr(args, "generation_batch_size", None)
                if gen_bs is not None:
                    hp_info["generation_batch_size"] = gen_bs
            except Exception:
                pass

            # 3. ETA calculation
            elapsed = 0
            eta = None
            if self._train_start_time and state.max_steps > 0:
                elapsed = round(time.time() - self._train_start_time, 1)
                current_step = state.global_step if hasattr(state, 'global_step') else 0
                if current_step > 0:
                    secs_per_step = elapsed / current_step
                    remaining_steps = state.max_steps - current_step
                    eta = round(secs_per_step * remaining_steps, 1)

            # 3b. Epoch
            current_epoch = round(getattr(state, 'epoch', 0) or 0, 2)

            # 4. Build enriched log entry from current logs dict
            log_entry = {}
            if logs:
                step = state.global_step if hasattr(state, 'global_step') else 0
                log_entry["step"] = step

                # Standard metrics (all task types)
                for key in ("loss", "eval_loss", "learning_rate", "grad_norm",
                            "train_samples_per_second", "train_steps_per_second"):
                    if key in logs:
                        log_entry[key] = logs[key]

                # Tokens/sec — derive from samples*seq_len or use direct key
                if "train_samples_per_second" in logs:
                    # Approximate: multiply by max_seq_length if available
                    seq_len = getattr(args, "max_seq_length", None) or getattr(args, "max_length", None)
                    if seq_len:
                        log_entry["tokens_per_sec"] = round(logs["train_samples_per_second"] * seq_len, 1)
                if "tokens_per_sec" in logs:
                    log_entry["tokens_per_sec"] = logs["tokens_per_sec"]

                # DPO-specific
                for key in ("rewards/chosen", "rewards/rejected",
                            "rewards/accuracies", "rewards/margins",
                            "logps/chosen", "logps/rejected",
                            "kl", "kl_divergence"):
                    if key in logs:
                        # Normalise key names for the frontend
                        clean = key.replace("/", "_")
                        log_entry[clean] = logs[key]

                # GRPO-specific
                for key in ("reward", "reward_std", "kl", "kl_divergence",
                            "completion_length", "policy_loss", "value_loss"):
                    if key in logs:
                        log_entry[key] = logs[key]

            # 5. Overwrite the global payload (no disk I/O)
            existing_logs = _GLOBAL_PAYLOAD.get("logs", [])
            # Merge into existing step entry if present, otherwise append
            step_key = log_entry.get("step", -1)
            merged = False
            for entry in existing_logs:
                if entry.get("step") == step_key:
                    entry.update(log_entry)
                    merged = True
                    break
            if not merged and log_entry:
                existing_logs.append(log_entry)

            # Live memory breakdown (unsloth-studio style)
            try:
                import torch
                peak_mb = None
                if torch.cuda.is_available():
                    peak_mb  = int(torch.cuda.max_memory_reserved() / (1024 * 1024))
                elif torch.backends.mps.is_available():
                    peak_mb  = int(torch.mps.driver_allocated_memory() / (1024 * 1024))
                if peak_mb is not None:
                    lora_mb  = peak_mb - self._baseline_vram_mb
                    total_mb = self._total_vram_mb or 1
                    hw_info.update({
                        "baseline_vram_mb": self._baseline_vram_mb,
                        "lora_vram_mb":     max(lora_mb, 0),
                        "total_vram_mb":    self._total_vram_mb,
                        "vram_pct":         round(peak_mb / total_mb * 100, 1),
                        "lora_vram_pct":    round(max(lora_mb, 0) / total_mb * 100, 1),
                    })
            except Exception:
                pass

            _GLOBAL_PAYLOAD = {
                "meta": {
                    "max_steps":    state.max_steps,
                    "total_epochs": getattr(args, "num_train_epochs", 0),
                    "task_type":    self.task_type,
                    "current_epoch": current_epoch,
                },
                "hardware": hw_info,
                "hyperparameters": hp_info,
                "logs": existing_logs,
                "phase": "training",
                "elapsed_seconds": elapsed,
                "eta_seconds": eta,
            }

            # 6. Push to all SSE subscribers instantly
            _notify_subscribers()


In [ ]:
from pathlib import Path

Path("templates").mkdir(exist_ok=True)
Path("templates/dashboard.html").write_text('<!DOCTYPE html>\n<html lang="en">\n\n<head>\n  <meta charset="UTF-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <title>Gaslamp | Training Dashboard</title>\n  <script src="https://cdn.tailwindcss.com"></script>\n  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>\n  <style>\n    @import url(\'https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap\');\n\n    body {\n      font-family: \'Inter\', sans-serif;\n      background-color: #0f1115;\n      color: #e5e7eb;\n    }\n\n    .glass-panel {\n      background: rgba(30, 33, 40, 0.7);\n      backdrop-filter: blur(12px);\n      -webkit-backdrop-filter: blur(12px);\n      border: 1px solid rgba(255, 255, 255, 0.05);\n      border-radius: 1rem;\n      box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.3), 0 2px 4px -1px rgba(0, 0, 0, 0.18);\n    }\n\n    .gaslamp-gradient {\n      background: linear-gradient(135deg, #10b981 0%, #3b82f6 100%);\n      -webkit-background-clip: text;\n      background-clip: text;\n      -webkit-text-fill-color: transparent;\n    }\n\n    .metric-value {\n      font-variant-numeric: tabular-nums;\n    }\n\n    /* Task-type badge colours */\n    .badge-sft     { background: rgba(16,185,129,0.15); border-color: rgba(16,185,129,0.4);  color: #10b981; }\n    .badge-dpo     { background: rgba(59,130,246,0.15); border-color: rgba(59,130,246,0.4);  color: #3b82f6; }\n    .badge-grpo    { background: rgba(168,85,247,0.15); border-color: rgba(168,85,247,0.4);  color: #a855f7; }\n    .badge-vision  { background: rgba(245,158,11,0.15); border-color: rgba(245,158,11,0.4);  color: #f59e0b; }\n\n    /* Phase badge */\n    .phase-idle      { background:rgba(107,114,128,.15); border-color:rgba(107,114,128,.4); color:#9ca3af; }\n    .phase-training  { background:rgba(16,185,129,.15);  border-color:rgba(16,185,129,.4);  color:#10b981; }\n    .phase-completed { background:rgba(59,130,246,.15);  border-color:rgba(59,130,246,.4);  color:#3b82f6; }\n    .phase-error     { background:rgba(239,68,68,.15);   border-color:rgba(239,68,68,.4);   color:#ef4444; }\n\n    .pulse-dot {\n      display: inline-block;\n      width: 8px; height: 8px;\n      border-radius: 50%;\n      background: #10b981;\n      animation: pulse 1.5s ease-in-out infinite;\n      margin-right: 6px;\n    }\n\n    @keyframes pulse {\n      0%, 100% { opacity: 1; transform: scale(1); }\n      50%       { opacity: .4; transform: scale(.7); }\n    }\n\n    ::-webkit-scrollbar { height: 8px; width: 8px; }\n    ::-webkit-scrollbar-track  { background: rgba(0,0,0,.2); border-radius: 4px; }\n    ::-webkit-scrollbar-thumb  { background: rgba(255,255,255,.1); border-radius: 4px; }\n    ::-webkit-scrollbar-thumb:hover { background: rgba(255,255,255,.2); }\n\n    /* Smooth card fade-in */\n    .card-fadein {\n      animation: fadeIn .4s ease;\n    }\n    @keyframes fadeIn { from { opacity: 0; transform: translateY(8px); } to { opacity: 1; transform: translateY(0); } }\n\n    .toggle-btn {\n      padding: 2px 10px;\n      border-radius: 6px;\n      font-size: 0.7rem;\n      font-weight: 600;\n      cursor: pointer;\n      border: 1px solid rgba(255,255,255,.15);\n      background: rgba(255,255,255,.04);\n      color: #9ca3af;\n      transition: all .2s;\n    }\n    .toggle-btn:hover { background: rgba(255,255,255,.1); color: #e5e7eb; }\n    .toggle-btn.active { background: rgba(16,185,129,.2); border-color: rgba(16,185,129,.5); color: #10b981; }\n  </style>\n</head>\n\n<body class="min-h-screen p-6 md:p-8">\n\n  <!-- ── Header ─────────────────────────────────────────────────── -->\n  <header class="mb-8">\n    <div class="flex flex-col md:flex-row md:items-center justify-between gap-4 border-b border-gray-800 pb-6 mb-6">\n      <div class="flex items-center space-x-4">\n        <div class="w-14 h-14 rounded-xl bg-gray-900 border border-gray-700 shadow-lg shadow-emerald-500/20 flex items-center justify-center p-1">\n          <svg viewBox="0 0 64 64" fill="none" xmlns="http://www.w3.org/2000/svg" class="w-10 h-10">\n            <circle cx="32" cy="32" r="30" fill="url(#g1)" opacity=".15"/>\n            <path d="M20 44 L32 20 L44 44" stroke="url(#g1)" stroke-width="3" stroke-linecap="round" stroke-linejoin="round"/>\n            <circle cx="32" cy="20" r="3" fill="#10b981"/>\n            <defs>\n              <linearGradient id="g1" x1="0" y1="0" x2="64" y2="64" gradientUnits="userSpaceOnUse">\n                <stop offset="0%" stop-color="#10b981"/>\n                <stop offset="100%" stop-color="#3b82f6"/>\n              </linearGradient>\n            </defs>\n          </svg>\n        </div>\n        <div>\n          <h1 class="text-2xl font-bold gaslamp-gradient">Gaslamp</h1>\n          <p class="text-gray-500 text-sm">Live Training Dashboard</p>\n        </div>\n      </div>\n      <div class="flex flex-wrap items-center gap-3">\n        <!-- Task-type badge -->\n        <span id="task-badge" class="px-3 py-1 rounded-full text-xs font-semibold border badge-sft">SFT</span>\n        <!-- Phase badge -->\n        <span id="phase-badge" class="px-3 py-1 rounded-full text-xs font-semibold border phase-idle">\n          <span class="pulse-dot hidden" id="pulse-dot"></span>\n          <span id="phase-text">IDLE</span>\n        </span>\n        <!-- EMA toggle -->\n        <button id="ema-toggle" class="toggle-btn active" onclick="toggleEma()">EMA</button>\n        <!-- Connection indicator -->\n        <span id="conn-badge" class="px-3 py-1 rounded-full text-xs font-semibold border border-gray-700 text-gray-500">⏳ connecting</span>\n      </div>\n    </div>\n\n    <!-- Progress bar -->\n    <div class="glass-panel p-4">\n      <div class="flex justify-between text-xs text-gray-400 mb-2">\n        <span id="progress-label">Step 0 / ?</span>\n        <span id="epoch-label">Epoch — / —</span>\n      </div>\n      <div class="w-full bg-gray-800 rounded-full h-2.5 overflow-hidden mb-2">\n        <div id="progress-bar" class="h-2.5 rounded-full transition-all duration-500"\n             style="width: 0%; background: linear-gradient(90deg,#10b981,#3b82f6);"></div>\n      </div>\n      <!-- Epoch sub-progress -->\n      <div class="flex justify-between text-xs text-gray-600 mb-1">\n        <span>Epoch progress</span>\n        <span id="epoch-pct">0%</span>\n      </div>\n      <div class="w-full bg-gray-800/60 rounded-full h-1.5 overflow-hidden">\n        <div id="epoch-bar" class="h-1.5 rounded-full transition-all duration-500"\n             style="width: 0%; background: linear-gradient(90deg,#a855f7,#3b82f6);"></div>\n      </div>\n    </div>\n  </header>\n\n  <!-- ── Stat Cards Row ──────────────────────────────────────────── -->\n  <div class="grid grid-cols-2 md:grid-cols-4 gap-4 mb-6" id="stat-cards">\n    <!-- ETA -->\n    <div class="glass-panel p-4">\n      <p class="text-xs text-gray-400 mb-1">⏳ ETA</p>\n      <p class="text-xl font-bold metric-value" id="stat-eta">—</p>\n    </div>\n    <!-- Elapsed -->\n    <div class="glass-panel p-4">\n      <p class="text-xs text-gray-400 mb-1">🕐 Elapsed</p>\n      <p class="text-xl font-bold metric-value" id="stat-elapsed">—</p>\n    </div>\n    <!-- Peak VRAM -->\n    <div class="glass-panel p-4">\n      <p class="text-xs text-gray-400 mb-1">🎮 Peak VRAM</p>\n      <p class="text-xl font-bold metric-value" id="stat-vram">—</p>\n    </div>\n    <!-- Tokens/sec (shown when available) -->\n    <div class="glass-panel p-4" id="card-tps">\n      <p class="text-xs text-gray-400 mb-1">⚡ Tokens/sec</p>\n      <p class="text-xl font-bold metric-value" id="stat-tps">—</p>\n    </div>\n  </div>\n\n  <!-- Optional stat cards: grad norm + CPU RAM -->\n  <div class="grid grid-cols-2 gap-4 mb-6 hidden" id="extra-stat-row">\n    <div class="glass-panel p-4">\n      <p class="text-xs text-gray-400 mb-1">📐 Grad Norm</p>\n      <p class="text-xl font-bold metric-value" id="stat-grad-norm">—</p>\n    </div>\n    <div class="glass-panel p-4">\n      <p class="text-xs text-gray-400 mb-1">💾 CPU RAM</p>\n      <p class="text-xl font-bold metric-value" id="stat-cpu-ram">—</p>\n    </div>\n  </div>\n\n  <!-- Memory Breakdown Card (shown when total_vram available) -->\n  <div class="glass-panel p-5 mb-6 hidden" id="mem-breakdown">\n    <h2 class="font-semibold text-sm text-gray-300 mb-3">📦 GPU Memory Breakdown <span class="text-xs text-gray-500 font-normal ml-1">(Unsloth-style)</span></h2>\n    <div class="grid grid-cols-2 md:grid-cols-4 gap-4">\n      <div>\n        <p class="text-xs text-gray-500 mb-1">Total GPU</p>\n        <p class="text-lg font-bold metric-value" id="mem-total">—</p>\n      </div>\n      <div>\n        <p class="text-xs text-gray-500 mb-1">Model Baseline</p>\n        <p class="text-lg font-bold metric-value" id="mem-baseline">—</p>\n      </div>\n      <div>\n        <p class="text-xs text-gray-500 mb-1">LoRA Delta</p>\n        <p class="text-lg font-bold text-emerald-400 metric-value" id="mem-lora">—</p>\n      </div>\n      <div>\n        <p class="text-xs text-gray-500 mb-1">Peak / Total</p>\n        <p class="text-lg font-bold metric-value" id="mem-pct">—</p>\n      </div>\n    </div>\n    <!-- VRAM gauge bars -->\n    <div class="mt-4 space-y-2">\n      <div>\n        <div class="flex justify-between text-xs text-gray-500 mb-1">\n          <span>Peak VRAM usage</span><span id="mem-pct-label">0%</span>\n        </div>\n        <div class="w-full bg-gray-800 rounded-full h-2 overflow-hidden">\n          <div id="mem-bar-peak" class="h-2 rounded-full transition-all duration-500" style="width:0%; background:linear-gradient(90deg,#10b981,#3b82f6);"></div>\n        </div>\n      </div>\n      <div>\n        <div class="flex justify-between text-xs text-gray-500 mb-1">\n          <span>LoRA training overhead</span><span id="mem-lora-pct-label">0%</span>\n        </div>\n        <div class="w-full bg-gray-800 rounded-full h-2 overflow-hidden">\n          <div id="mem-bar-lora" class="h-2 rounded-full transition-all duration-500" style="width:0%; background:linear-gradient(90deg,#a855f7,#3b82f6);"></div>\n        </div>\n      </div>\n    </div>\n  </div>\n\n  <!-- Completed summary panel -->\n  <div id="completed-summary" class="glass-panel p-5 mb-6 hidden border border-emerald-900">\n    <h2 class="font-semibold text-sm text-emerald-400 mb-3">✅ Training Complete — Summary</h2>\n    <div class="grid grid-cols-2 md:grid-cols-3 gap-y-3 gap-x-6 text-xs" id="completed-grid"></div>\n  </div>\n\n  <!-- ── Main Charts ─────────────────────────────────────────────── -->\n  <div class="grid grid-cols-1 lg:grid-cols-2 gap-6 mb-6">\n\n    <!-- Loss + LR dual-axis -->\n    <div class="glass-panel p-5 card-fadein">\n      <div class="flex items-center justify-between mb-3">\n        <h2 class="font-semibold text-sm text-gray-200">📉 Training Loss</h2>\n        <div class="flex gap-2">\n          <span class="text-xs text-gray-400" id="loss-latest">—</span>\n        </div>\n      </div>\n      <canvas id="chart-loss" height="180"></canvas>\n    </div>\n\n    <!-- Eval Loss -->\n    <div class="glass-panel p-5 card-fadein">\n      <div class="flex items-center justify-between mb-3">\n        <h2 class="font-semibold text-sm text-gray-200">🧪 Evaluation Loss</h2>\n        <span class="text-xs text-gray-400" id="eval-latest">No eval data yet</span>\n      </div>\n      <canvas id="chart-eval" height="180"></canvas>\n    </div>\n\n    <!-- Learning Rate -->\n    <div class="glass-panel p-5 card-fadein">\n      <div class="flex items-center justify-between mb-3">\n        <h2 class="font-semibold text-sm text-gray-200">📈 Learning Rate</h2>\n        <span class="text-xs text-gray-400" id="lr-latest">—</span>\n      </div>\n      <canvas id="chart-lr" height="180"></canvas>\n    </div>\n\n    <!-- Grad Norm (always rendered, fades in when data arrives) -->\n    <div class="glass-panel p-5 card-fadein">\n      <div class="flex items-center justify-between mb-3">\n        <h2 class="font-semibold text-sm text-gray-200">📐 Gradient Norm</h2>\n        <span class="text-xs text-gray-400" id="gn-latest">—</span>\n      </div>\n      <canvas id="chart-gn" height="180"></canvas>\n    </div>\n\n  </div>\n\n  <!-- ── Task-specific Panels ────────────────────────────────────── -->\n\n  <!-- GRPO: Reward + KL -->\n  <div id="grpo-panels" class="grid grid-cols-1 lg:grid-cols-2 gap-6 mb-6 hidden">\n    <div class="glass-panel p-5 card-fadein">\n      <div class="flex items-center justify-between mb-3">\n        <h2 class="font-semibold text-sm text-gray-200">🎯 GRPO Reward</h2>\n        <span class="text-xs text-gray-400" id="reward-latest">—</span>\n      </div>\n      <canvas id="chart-reward" height="180"></canvas>\n    </div>\n    <div class="glass-panel p-5 card-fadein">\n      <div class="flex items-center justify-between mb-3">\n        <h2 class="font-semibold text-sm text-gray-200">🔀 KL Divergence</h2>\n        <span class="text-xs text-gray-400" id="kl-latest">—</span>\n      </div>\n      <canvas id="chart-kl" height="180"></canvas>\n    </div>\n  </div>\n\n  <!-- DPO: Chosen/Rejected + KL -->\n  <div id="dpo-panels" class="grid grid-cols-1 lg:grid-cols-2 gap-6 mb-6 hidden">\n    <div class="glass-panel p-5 card-fadein">\n      <div class="flex items-center justify-between mb-3">\n        <h2 class="font-semibold text-sm text-gray-200">⚖️ Chosen vs Rejected Reward</h2>\n        <span class="text-xs text-gray-400" id="dpo-reward-latest">—</span>\n      </div>\n      <canvas id="chart-dpo-reward" height="180"></canvas>\n    </div>\n    <div class="glass-panel p-5 card-fadein">\n      <div class="flex items-center justify-between mb-3">\n        <h2 class="font-semibold text-sm text-gray-200">🔀 KL Divergence</h2>\n        <span class="text-xs text-gray-400" id="dpo-kl-latest">—</span>\n      </div>\n      <canvas id="chart-dpo-kl" height="180"></canvas>\n    </div>\n  </div>\n\n  <!-- Vision SFT placeholder badge -->\n  <div id="vision-panel" class="mb-6 hidden">\n    <div class="glass-panel p-5 flex items-center gap-4">\n      <span class="text-3xl">🖼️</span>\n      <div>\n        <h2 class="font-semibold text-sm text-gray-200">Vision SFT Mode</h2>\n        <p class="text-xs text-gray-500 mt-1">Training a Vision Language Model. Standard loss and eval metrics apply.</p>\n      </div>\n    </div>\n  </div>\n\n  <!-- ── Hyperparameters & Hardware ─────────────────────────────── -->\n  <div class="grid grid-cols-1 md:grid-cols-2 gap-6 mb-6">\n    <div class="glass-panel p-5">\n      <h2 class="font-semibold text-sm text-gray-300 mb-3">⚙️ Hyperparameters</h2>\n      <div id="hp-grid" class="grid grid-cols-2 gap-y-2 gap-x-4 text-xs"></div>\n    </div>\n    <div class="glass-panel p-5">\n      <h2 class="font-semibold text-sm text-gray-300 mb-3">🖥️ Hardware</h2>\n      <div id="hw-grid" class="grid grid-cols-2 gap-y-2 gap-x-4 text-xs"></div>\n    </div>\n  </div>\n\n  <!-- ── Log Table ──────────────────────────────────────────────── -->\n  <div class="glass-panel p-5 mb-6">\n    <h2 class="font-semibold text-sm text-gray-300 mb-3">📋 Recent Log Entries</h2>\n    <div class="overflow-x-auto">\n      <table class="w-full text-xs text-left">\n        <thead>\n          <tr class="text-gray-500 border-b border-gray-800">\n            <th class="pb-2 pr-4">Step</th>\n            <th class="pb-2 pr-4">Loss</th>\n            <th class="pb-2 pr-4">Eval Loss</th>\n            <th class="pb-2 pr-4">LR</th>\n            <th class="pb-2 pr-4">Grad Norm</th>\n            <th class="pb-2 pr-4">Tokens/s</th>\n            <th class="pb-2 pr-4 dpo-col grpo-col hidden">Reward</th>\n            <th class="pb-2 dpo-col grpo-col hidden">KL</th>\n          </tr>\n        </thead>\n        <tbody id="log-tbody" class="text-gray-300 divide-y divide-gray-800/50"></tbody>\n      </table>\n    </div>\n  </div>\n\n  <footer class="text-center text-xs text-gray-600 pb-4">\n    Gaslamp Training Dashboard · Powered by Unsloth-buddy ·\n    <a href="https://github.com/TYH-labs/unsloth-buddy" class="underline hover:text-gray-400">GitHub</a>\n  </footer>\n\n<!-- ── JavaScript ──────────────────────────────────────────────── -->\n<script>\n// ─── Shared Chart Config ──────────────────────────────────────────\nconst CHART_DEFAULTS = {\n  responsive: true,\n  animation: { duration: 300 },\n  plugins: {\n    legend: { labels: { color: \'#9ca3af\', font: { size: 11 } } },\n    tooltip: { mode: \'index\', intersect: false },\n  },\n  scales: {\n    x: { ticks: { color: \'#6b7280\', font: { size: 10 } }, grid: { color: \'rgba(255,255,255,.04)\' } },\n    y: { ticks: { color: \'#6b7280\', font: { size: 10 } }, grid: { color: \'rgba(255,255,255,.04)\' } },\n  },\n};\n\nconst COLORS = {\n  emerald:   { border: \'rgba(16,185,129,1)\', bg: \'rgba(16,185,129,.12)\' },\n  blue:      { border: \'rgba(59,130,246,1)\', bg: \'rgba(59,130,246,.12)\' },\n  purple:    { border: \'rgba(168,85,247,1)\', bg: \'rgba(168,85,247,.12)\' },\n  orange:    { border: \'rgba(249,115,22,1)\', bg: \'rgba(249,115,22,.12)\' },\n  rose:      { border: \'rgba(244,63,94,1)\',  bg: \'rgba(244,63,94,.12)\' },\n  amber:     { border: \'rgba(245,158,11,1)\', bg: \'rgba(245,158,11,.12)\' },\n  cyan:      { border: \'rgba(6,182,212,1)\',  bg: \'rgba(6,182,212,.12)\' },\n  gray:      { border: \'rgba(107,114,128,.6)\', bg: \'rgba(107,114,128,.06)\' },\n};\n\nfunction lineDataset(label, col, data, tension=0.35) {\n  return {\n    label,\n    data,\n    borderColor: COLORS[col].border,\n    backgroundColor: COLORS[col].bg,\n    borderWidth: 2,\n    pointRadius: 2,\n    pointHoverRadius: 4,\n    tension,\n    fill: false,\n  };\n}\n\nfunction makeChart(id, datasets, yLabel=\'\', rightDatasets=null) {\n  const ctx = document.getElementById(id).getContext(\'2d\');\n  const scales = {\n    x: CHART_DEFAULTS.scales.x,\n    y: { ...CHART_DEFAULTS.scales.y, title: { display: !!yLabel, text: yLabel, color: \'#6b7280\', font:{size:10} } },\n  };\n  if (rightDatasets) {\n    scales.y2 = {\n      type: \'linear\',\n      position: \'right\',\n      ticks: { color: \'#6b7280\', font: { size: 10 } },\n      grid: { drawOnChartArea: false },\n    };\n    rightDatasets.forEach(d => d.yAxisID = \'y2\');\n  }\n  return new Chart(ctx, {\n    type: \'line\',\n    data: { labels: [], datasets: rightDatasets ? [...datasets, ...rightDatasets] : datasets },\n    options: { ...CHART_DEFAULTS, scales },\n  });\n}\n\n// ─── Chart instances ──────────────────────────────────────────────\nlet showEma = true;\n\nconst chartLoss   = makeChart(\'chart-loss\',   [lineDataset(\'Train Loss\',\'emerald\',[]), lineDataset(\'EMA Loss\',\'blue\',[])]);\nconst chartEval   = makeChart(\'chart-eval\',   [lineDataset(\'Eval Loss\',\'purple\',[])]);\nconst chartLR     = makeChart(\'chart-lr\',     [lineDataset(\'Learning Rate\',\'amber\',[])]);\nconst chartGN     = makeChart(\'chart-gn\',     [lineDataset(\'Grad Norm\',\'cyan\',[])]);\nconst chartReward = makeChart(\'chart-reward\', [lineDataset(\'Reward\',\'emerald\',[]), lineDataset(\'Reward+σ\',\'gray\',[]), lineDataset(\'Reward−σ\',\'gray\',[])]);\nconst chartKL     = makeChart(\'chart-kl\',     [lineDataset(\'KL Div\',\'purple\',[])]);\nconst chartDpoRew = makeChart(\'chart-dpo-reward\', [lineDataset(\'Chosen\',\'emerald\',[]), lineDataset(\'Rejected\',\'rose\',[])]);\nconst chartDpoKL  = makeChart(\'chart-dpo-kl\',    [lineDataset(\'KL Div\',\'purple\',[])]);\n\nfunction toggleEma() {\n  showEma = !showEma;\n  document.getElementById(\'ema-toggle\').classList.toggle(\'active\', showEma);\n  const emaDs = chartLoss.data.datasets[1];\n  emaDs.hidden = !showEma;\n  chartLoss.update(\'none\');\n}\n\n// ─── EMA helper ───────────────────────────────────────────────────\nfunction ema(arr, alpha=0.1) {\n  if (!arr.length) return [];\n  const out = [arr[0]];\n  for (let i=1; i<arr.length; i++) out.push(alpha*arr[i]+(1-alpha)*out[i-1]);\n  return out;\n}\n\nfunction fmt(v) { return (typeof v === \'number\') ? v.toFixed(4) : \'—\'; }\nfunction fmtSec(s) {\n  if (!s && s !== 0) return \'—\';\n  const h=Math.floor(s/3600), m=Math.floor((s%3600)/60), sec=Math.floor(s%60);\n  return h>0 ? `${h}h ${m}m ${sec}s` : m>0 ? `${m}m ${sec}s` : `${sec}s`;\n}\n\n// ─── Update charts from payload ───────────────────────────────────\nfunction updateCharts(payload) {\n  const logs = payload.logs || [];\n  const meta = payload.meta || {};\n  const hw   = payload.hardware || {};\n  const hp   = payload.hyperparameters || {};\n  const task = (meta.task_type || \'sft\').toLowerCase();\n\n  // ── Phase badge ──\n  const phase = (payload.phase || \'idle\').toLowerCase();\n  const phaseBadge = document.getElementById(\'phase-badge\');\n  const phaseTxt   = document.getElementById(\'phase-text\');\n  const pulseDot   = document.getElementById(\'pulse-dot\');\n  phaseBadge.className = `px-3 py-1 rounded-full text-xs font-semibold border phase-${phase}`;\n  phaseTxt.textContent  = phase.toUpperCase();\n  pulseDot.classList.toggle(\'hidden\', phase !== \'training\');\n\n  // ── Task-type badge ──\n  const taskBadge = document.getElementById(\'task-badge\');\n  taskBadge.className = `px-3 py-1 rounded-full text-xs font-semibold border badge-${task}`;\n  taskBadge.textContent = task.toUpperCase();\n\n  // ── Task-specific panel visibility ──\n  document.getElementById(\'grpo-panels\').classList.toggle(\'hidden\', task !== \'grpo\');\n  document.getElementById(\'dpo-panels\').classList.toggle(\'hidden\',  task !== \'dpo\');\n  document.getElementById(\'vision-panel\').classList.toggle(\'hidden\', task !== \'vision\');\n  // Show task-specific log table columns\n  document.querySelectorAll(\'.dpo-col\').forEach(el => el.classList.toggle(\'hidden\', task !== \'dpo\'));\n  document.querySelectorAll(\'.grpo-col\').forEach(el => el.classList.toggle(\'hidden\', task !== \'grpo\'));\n\n  // ── Progress bar ──\n  const maxSteps = meta.max_steps || 0;\n  const lastStep = logs.length ? (logs[logs.length-1].step || 0) : 0;\n  const pct = maxSteps > 0 ? Math.min(100, Math.round(lastStep/maxSteps*100)) : 0;\n  document.getElementById(\'progress-bar\').style.width = pct + \'%\';\n  document.getElementById(\'progress-label\').textContent =\n    `Step ${lastStep} / ${maxSteps || \'?\'}  (${pct}%)`;\n  const epochs = meta.total_epochs || \'?\';\n  document.getElementById(\'epoch-label\').textContent = `Epochs: ${epochs}`;\n\n  // Epoch sub-bar\n  const curEpoch  = meta.current_epoch || 0;\n  const totEpochs = meta.total_epochs  || 1;\n  const epochPct  = Math.min(100, Math.round((curEpoch / totEpochs) * 100));\n  document.getElementById(\'epoch-bar\').style.width = epochPct + \'%\';\n  document.getElementById(\'epoch-pct\').textContent  = epochPct + \'%\';\n  document.getElementById(\'epoch-label\').textContent = `Epoch ${curEpoch.toFixed(2)} / ${totEpochs}`;\n\n  // ── Stat cards ──\n  document.getElementById(\'stat-eta\').textContent     = fmtSec(payload.eta_seconds);\n  document.getElementById(\'stat-elapsed\').textContent = fmtSec(payload.elapsed_seconds);\n  document.getElementById(\'stat-vram\').textContent    = hw.peak_vram_mb ? `${hw.peak_vram_mb} MB` : \'—\';\n\n  // Grad norm + CPU RAM row\n  const lastGN  = [...logs].reverse().find(l => \'grad_norm\'  in l);\n  const lastTPS = [...logs].reverse().find(l => \'tokens_per_sec\' in l || \'it_per_sec\' in l);\n  if (lastGN || hw.cpu_ram_mb) {\n    document.getElementById(\'extra-stat-row\').classList.remove(\'hidden\');\n    document.getElementById(\'stat-grad-norm\').textContent = lastGN ? fmt(lastGN.grad_norm) : \'—\';\n    document.getElementById(\'stat-cpu-ram\').textContent   = hw.cpu_ram_mb ? `${hw.cpu_ram_mb} MB` : \'—\';\n  }\n  if (lastTPS) {\n    const v = lastTPS.tokens_per_sec || lastTPS.it_per_sec;\n    document.getElementById(\'stat-tps\').textContent = typeof v === \'number\' ? v.toFixed(1) : \'—\';\n  }\n\n  // ── Memory Breakdown Panel ──\n  if (hw.total_vram_mb) {\n    document.getElementById(\'mem-breakdown\').classList.remove(\'hidden\');\n    const fmtMb = mb => `${mb} MB (${(mb/1024).toFixed(2)} GB)`;\n    document.getElementById(\'mem-total\').textContent    = fmtMb(hw.total_vram_mb);\n    document.getElementById(\'mem-baseline\').textContent = fmtMb(hw.baseline_vram_mb || 0);\n    document.getElementById(\'mem-lora\').textContent     = fmtMb(hw.lora_vram_mb || 0);\n    document.getElementById(\'mem-pct\').textContent      = `${hw.vram_pct || 0}%`;\n    // Gauge bars\n    const vPct = Math.min(hw.vram_pct || 0, 100);\n    const lPct = Math.min(hw.lora_vram_pct || 0, 100);\n    document.getElementById(\'mem-bar-peak\').style.width  = vPct + \'%\';\n    document.getElementById(\'mem-bar-lora\').style.width  = lPct + \'%\';\n    document.getElementById(\'mem-pct-label\').textContent      = vPct + \'%\';\n    document.getElementById(\'mem-lora-pct-label\').textContent = lPct + \'%\';\n  }\n\n  // ── Series extraction ──\n  const trainSteps=[], trainLoss=[], evalSteps=[], evalLoss=[];\n  const lrSteps=[],  lrs=[];\n  const gnSteps=[],  gns=[];\n  const rewSteps=[], rewards=[], rewHi=[], rewLo=[];\n  const klSteps=[],  kls=[];\n  const choSteps=[], choVals=[], rejSteps=[], rejVals=[];\n\n  for (const log of logs) {\n    const s = log.step;\n    if (s === undefined) continue;\n    if (\'loss\' in log) { trainSteps.push(s); trainLoss.push(log.loss); }\n    if (\'eval_loss\' in log) { evalSteps.push(s); evalLoss.push(log.eval_loss); }\n    if (\'learning_rate\' in log) { lrSteps.push(s); lrs.push(log.learning_rate); }\n    if (\'grad_norm\' in log) { gnSteps.push(s); gns.push(log.grad_norm); }\n    if (\'reward\' in log) {\n      rewSteps.push(s);\n      rewards.push(log.reward);\n      const std = log.reward_std || 0;\n      rewHi.push(log.reward + std);\n      rewLo.push(log.reward - std);\n    }\n    const kl = log.kl_divergence ?? log.kl;\n    if (kl !== undefined) { klSteps.push(s); kls.push(kl); }\n    if (\'rewards_chosen\' in log)   { choSteps.push(s); choVals.push(log.rewards_chosen); }\n    if (\'rewards_rejected\' in log) { rejSteps.push(s); rejVals.push(log.rewards_rejected); }\n  }\n\n  // ── Loss chart ──\n  const smoothed = ema(trainLoss);\n  chartLoss.data.labels = trainSteps;\n  chartLoss.data.datasets[0].data = trainLoss;\n  chartLoss.data.datasets[1].data = smoothed;\n  chartLoss.data.datasets[1].hidden = !showEma;\n  chartLoss.update(\'none\');\n  document.getElementById(\'loss-latest\').textContent =\n    trainLoss.length ? `latest: ${trainLoss[trainLoss.length-1].toFixed(4)}` : \'\';\n\n  // ── Eval chart ──\n  chartEval.data.labels = evalSteps;\n  chartEval.data.datasets[0].data = evalLoss;\n  chartEval.update(\'none\');\n  document.getElementById(\'eval-latest\').textContent =\n    evalLoss.length ? `latest: ${evalLoss[evalLoss.length-1].toFixed(4)}` : \'No eval data yet\';\n\n  // ── LR chart ──\n  chartLR.data.labels = lrSteps;\n  chartLR.data.datasets[0].data = lrs;\n  chartLR.update(\'none\');\n  document.getElementById(\'lr-latest\').textContent =\n    lrs.length ? lrs[lrs.length-1].toExponential(3) : \'—\';\n\n  // ── Grad norm chart ──\n  chartGN.data.labels = gnSteps;\n  chartGN.data.datasets[0].data = gns;\n  chartGN.update(\'none\');\n  document.getElementById(\'gn-latest\').textContent =\n    gns.length ? gns[gns.length-1].toFixed(4) : \'—\';\n\n  // ── GRPO Reward ──\n  chartReward.data.labels = rewSteps;\n  chartReward.data.datasets[0].data = rewards;\n  chartReward.data.datasets[1].data = rewHi;\n  chartReward.data.datasets[2].data = rewLo;\n  chartReward.update(\'none\');\n  document.getElementById(\'reward-latest\').textContent =\n    rewards.length ? `latest: ${rewards[rewards.length-1].toFixed(4)}` : \'—\';\n\n  // ── GRPO KL ──\n  chartKL.data.labels = klSteps;\n  chartKL.data.datasets[0].data = kls;\n  chartKL.update(\'none\');\n  document.getElementById(\'kl-latest\').textContent =\n    kls.length ? kls[kls.length-1].toFixed(4) : \'—\';\n\n  // ── DPO chosen/rejected ──\n  chartDpoRew.data.labels = choSteps.length >= rejSteps.length ? choSteps : rejSteps;\n  chartDpoRew.data.datasets[0].data = choVals;\n  chartDpoRew.data.datasets[1].data = rejVals;\n  chartDpoRew.update(\'none\');\n  document.getElementById(\'dpo-reward-latest\').textContent =\n    choVals.length ? `chosen: ${choVals[choVals.length-1].toFixed(4)}` : \'—\';\n\n  // ── DPO KL ──\n  chartDpoKL.data.labels = klSteps;\n  chartDpoKL.data.datasets[0].data = kls;\n  chartDpoKL.update(\'none\');\n  document.getElementById(\'dpo-kl-latest\').textContent =\n    kls.length ? kls[kls.length-1].toFixed(4) : \'—\';\n\n  // ── Hyperparameters panel ──\n  const hpGrid = document.getElementById(\'hp-grid\');\n  hpGrid.innerHTML = \'\';\n  const hpEntries = [\n    [\'Learning Rate\', hp.learning_rate !== undefined ? hp.learning_rate.toExponential(2) : \'—\'],\n    [\'Batch Size\', hp.train_batch_size ?? \'—\'],\n    [\'Grad Accumulation\', hp.gradient_accumulation ?? \'—\'],\n    [\'Optimizer\', hp.optimizer ?? \'—\'],\n    [\'Seed\', hp.seed ?? \'—\'],\n  ];\n  hpEntries.forEach(([k,v]) => {\n    hpGrid.insertAdjacentHTML(\'beforeend\',\n      `<span class="text-gray-500">${k}</span><span class="text-gray-200 font-medium">${v}</span>`);\n  });\n\n  // ── Hardware panel ──\n  const hwGrid = document.getElementById(\'hw-grid\');\n  hwGrid.innerHTML = \'\';\n  const hwEntries = [\n    [\'Device\', hw.device ?? \'—\'],\n    [\'Peak VRAM\', hw.peak_vram_mb ? `${hw.peak_vram_mb} MB` : \'—\'],\n  ];\n  if (hw.cpu_ram_mb) hwEntries.push([\'CPU RAM\', `${hw.cpu_ram_mb} MB`]);\n  hwEntries.forEach(([k,v]) => {\n    hwGrid.insertAdjacentHTML(\'beforeend\',\n      `<span class="text-gray-500">${k}</span><span class="text-gray-200 font-medium">${v}</span>`);\n  });\n\n  // ── Completed summary panel ──\n  const summaryPanel = document.getElementById(\'completed-summary\');\n  const summaryGrid  = document.getElementById(\'completed-grid\');\n  if (phase === \'completed\' && (payload.train_runtime_seconds || hw.total_vram_mb)) {\n    summaryPanel.classList.remove(\'hidden\');\n    summaryGrid.innerHTML = \'\';\n    const rt = payload.train_runtime_seconds || payload.elapsed_seconds || 0;\n    const rows = [\n      [\'Training time\', rt ? `${fmtSec(rt)} (${(rt/60).toFixed(2)} min)` : \'—\'],\n      [\'Total GPU memory\', hw.total_vram_mb ? `${hw.total_vram_mb} MB (${(hw.total_vram_mb/1024).toFixed(2)} GB)` : \'—\'],\n      [\'Peak reserved\',    hw.peak_vram_mb  ? `${hw.peak_vram_mb} MB — ${hw.vram_pct || 0}% of max` : \'—\'],\n      [\'Model baseline\',   hw.baseline_vram_mb !== undefined ? `${hw.baseline_vram_mb} MB` : \'—\'],\n      [\'LoRA overhead\',    hw.lora_vram_mb !== undefined ? `${hw.lora_vram_mb} MB — ${hw.lora_vram_pct || 0}% of max` : \'—\'],\n    ];\n    rows.forEach(([k, v]) => {\n      summaryGrid.insertAdjacentHTML(\'beforeend\',\n        `<span class="text-gray-500">${k}</span><span class="text-emerald-300 font-semibold">${v}</span>`);\n    });\n  } else {\n    summaryPanel.classList.add(\'hidden\');\n  }\n\n  // ── Log table (last 10 entries) ──\n  const tbody = document.getElementById(\'log-tbody\');\n  tbody.innerHTML = \'\';\n  const recent = [...logs].reverse().slice(0, 10);\n  for (const log of recent) {\n    const kl = log.kl_divergence ?? log.kl;\n    const rew = log.reward;\n    tbody.insertAdjacentHTML(\'beforeend\', `\n      <tr class="hover:bg-white/2 transition-colors">\n        <td class="py-1.5 pr-4 text-gray-400">${log.step ?? \'—\'}</td>\n        <td class="py-1.5 pr-4">${\'loss\' in log ? log.loss.toFixed(4) : \'—\'}</td>\n        <td class="py-1.5 pr-4">${\'eval_loss\' in log ? log.eval_loss.toFixed(4) : \'—\'}</td>\n        <td class="py-1.5 pr-4">${\'learning_rate\' in log ? log.learning_rate.toExponential(3) : \'—\'}</td>\n        <td class="py-1.5 pr-4">${\'grad_norm\' in log ? log.grad_norm.toFixed(4) : \'—\'}</td>\n        <td class="py-1.5 pr-4">${log.tokens_per_sec !== undefined ? log.tokens_per_sec.toFixed(1) : (log.it_per_sec !== undefined ? `${log.it_per_sec.toFixed(2)} it/s` : \'—\')}</td>\n        <td class="py-1.5 pr-4 dpo-col grpo-col hidden">${rew !== undefined ? rew.toFixed(4) : \'—\'}</td>\n        <td class="py-1.5 dpo-col grpo-col hidden">${kl !== undefined ? kl.toFixed(4) : \'—\'}</td>\n      </tr>`);\n  }\n}\n\n// ─── SSE Connection (with polling fallback) ───────────────────────\nlet lastPayload = null;\nlet pollTimer   = null;\n\nfunction startPollFallback(port=8080) {\n  // Poll every 10 s as a fallback only\n  pollTimer = setInterval(async () => {\n    try {\n      const r = await fetch(`http://localhost:${port}/api/metrics`);\n      if (r.ok) {\n        const payload = await r.json();\n        lastPayload = payload;\n        updateCharts(payload);\n      }\n    } catch(e) {}\n  }, 10000);\n}\n\nfunction connect(port=8080) {\n  const connBadge = document.getElementById(\'conn-badge\');\n  const es = new EventSource(`http://localhost:${port}/api/stream`);\n\n  es.addEventListener(\'progress\', (e) => {\n    try {\n      const payload = JSON.parse(e.data);\n      lastPayload = payload;\n      updateCharts(payload);\n      connBadge.textContent = \'🟢 live\';\n      connBadge.className   = \'px-3 py-1 rounded-full text-xs font-semibold border border-emerald-700 text-emerald-400\';\n    } catch(err) {}\n  });\n\n  es.onerror = () => {\n    connBadge.textContent = \'🟡 reconnecting\';\n    connBadge.className   = \'px-3 py-1 rounded-full text-xs font-semibold border border-amber-700 text-amber-400\';\n  };\n\n  es.onopen = () => {\n    // Also do an immediate HTTP fetch to get the full snapshot (handles reconnection)\n    fetch(`http://localhost:${port}/api/metrics`)\n      .then(r => r.ok ? r.json() : null)\n      .then(p => { if (p) { updateCharts(p); } })\n      .catch(()=>{});\n  };\n}\n\n// Detect port from URL hash, e.g. http://localhost:8080/#port=9090\nconst hashParams = new URLSearchParams(location.hash.slice(1));\nconst port = parseInt(hashParams.get(\'port\') || \'8080\', 10);\n\nconnect(port);\nstartPollFallback(port);\n</script>\n\n</body>\n</html>')
print("Training dashboard asset ready.")


In [ ]:
%%writefile train_ddp.py
"""Train one Qwen3-VL LoRA adapter on four Kaggle L4 GPUs."""

import os
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch


SEED = 3407
MAX_LENGTH = 8192
SMOKE_TEST = os.environ.get("ARC_VLM_SMOKE_TEST") == "1"
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
if WORLD_SIZE != 4:
    raise RuntimeError(f"Expected four DDP processes, got {WORLD_SIZE}")
torch.cuda.set_device(LOCAL_RANK)

from unsloth import FastVisionModel, UnslothVisionDataCollator
from datasets import load_from_disk
from trl import SFTConfig, SFTTrainer

from gaslamp_callback import GaslampDashboardCallback


model_path = Path("data/model_path.txt").read_text().strip()
model, processor = FastVisionModel.from_pretrained(
    model_name=model_path,
    max_seq_length=MAX_LENGTH,
    dtype=None,
    load_in_4bit=True,
    local_files_only=True,
    use_gradient_checkpointing="unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    target_modules="all-linear",
)
QWEN_EOS_TOKEN = "<|im_end|>"
if processor.tokenizer.convert_tokens_to_ids(QWEN_EOS_TOKEN) == processor.tokenizer.unk_token_id:
    raise RuntimeError(f"Model tokenizer is missing {QWEN_EOS_TOKEN}")
processor.tokenizer.eos_token = QWEN_EOS_TOKEN

training_args = SFTConfig(
    output_dir="outputs/checkpoints",
    logging_dir="outputs/logs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=2 if SMOKE_TEST else -1,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,
    bf16=True,
    fp16=False,
    logging_steps=5,
    save_strategy="epoch",
    report_to="tensorboard",
    seed=SEED,
    data_seed=SEED,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    packing=False,
    padding_free=False,
    ddp_find_unused_parameters=False,
)
training_args.eos_token = None

trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    train_dataset=load_from_disk("data/train"),
    data_collator=UnslothVisionDataCollator(
        model,
        processor,
        max_seq_length=MAX_LENGTH,
        resize=1024,
        resize_dimension="max",
        snap_to_patch_size=True,
        completion_only_loss=True,
    ),
    callbacks=[GaslampDashboardCallback(task_type="vision")],
    args=training_args,
)
trainer.train()
if trainer.is_world_process_zero():
    trainer.model.save_pretrained("outputs/adapters")
    processor.save_pretrained("outputs/adapters")


In [ ]:
import torch

assert torch.cuda.device_count() == 4, f"Select 4 x L4; found {torch.cuda.device_count()} GPU(s)"
!nvidia-smi -L
subprocess.run(["torchrun", "--standalone", "--nproc_per_node=4", "train_ddp.py"], check=True)


## Exact-grid validation

This loads the saved adapter and evaluates all 172 held-out queries greedily. It saves the raw generations, parsed predictions, and correctness flags to `outputs/eval_records.json`.


In [ ]:
%%writefile evaluate.py
"""Run exact-grid validation for the trained vision adapter."""

import json
import os
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")


def parse_output(text):
    start = text.find("{")
    if start < 0:
        raise ValueError("no JSON object")
    value, _ = json.JSONDecoder().raw_decode(text[start:])
    grid = value.get("output") if isinstance(value, dict) else None
    if not isinstance(grid, list) or not grid or any(not isinstance(row, list) for row in grid):
        raise ValueError("missing output grid")
    width = len(grid[0])
    if not 1 <= len(grid) <= 30 or not 1 <= width <= 30:
        raise ValueError("invalid grid size")
    if any(len(row) != width for row in grid):
        raise ValueError("non-rectangular grid")
    if any(type(value) is not int or not 0 <= value <= 9 for row in grid for value in row):
        raise ValueError("invalid color")
    return grid


def clean_messages(messages):
    return [{
        "role": message["role"],
        "content": [{key: value for key, value in item.items() if value is not None}
                    for item in message["content"]],
    } for message in messages]


def main():
    import torch
    from datasets import load_from_disk
    from unsloth import FastVisionModel

    model, processor = FastVisionModel.from_pretrained(
        model_name="outputs/adapters",
        max_seq_length=8192,
        dtype=None,
        load_in_4bit=True,
        local_files_only=True,
    )
    FastVisionModel.for_inference(model)

    @torch.inference_mode()
    def predict(row):
        messages = clean_messages(row["prompt"])
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=row["images"], return_tensors="pt").to(model.device)
        generated = model.generate(
            **inputs,
            max_new_tokens=128 if os.environ.get("ARC_VLM_SMOKE_TEST") == "1" else 2304,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
        return processor.tokenizer.decode(
            generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

    dataset = load_from_disk("data/eval")
    if os.environ.get("ARC_VLM_SMOKE_TEST") == "1":
        dataset = dataset.select(range(2))
    records = []
    for index, row in enumerate(dataset):
        text = predict(row)
        try:
            prediction = parse_output(text)
            error = None
        except (json.JSONDecodeError, TypeError, ValueError) as exc:
            prediction, error = None, str(exc)
        records.append({
            "task_id": row["task_id"],
            "episode_id": row["episode_id"],
            "correct": prediction == row["output"],
            "prediction": prediction,
            "expected": row["output"],
            "raw": text,
            "error": error,
        })
        if (index + 1) % 10 == 0:
            print(f"{index + 1}/{len(dataset)}")

    Path("outputs").mkdir(exist_ok=True)
    Path("outputs/eval_records.json").write_text(json.dumps(records))
    correct = sum(record["correct"] for record in records)
    print({"exact": correct, "total": len(records), "accuracy": correct / len(records)})


if __name__ == "__main__":
    main()


In [ ]:
!python evaluate.py

from pathlib import Path

assert Path("outputs/adapters/adapter_model.safetensors").exists()
assert Path("outputs/eval_records.json").exists()
print("Adapter and held-out predictions saved under outputs/.")
